# Course Review Lab: Palmer Penguins

**BA780 Introduction to Data Analytics**

This lab walks through the main tools of the course on one small dataset:

1. Load and inspect the data
2. Clean it (missing values, numbers stored as text)
3. Summarize with `groupby` and `pivot_table`
4. Visualize with Seaborn and Matplotlib subplots
5. Combine tables with `pd.merge`
6. Work with a time series
7. Build and evaluate one classification model

Sections 1 and 2 and the first cell of Section 3 review material from before Lecture 07, so their code is already filled in: run those cells, skim the output, and move on. Keep your code short and spend your time *reading* the output. Each remaining code cell has commented lines to guide you; replace the `# ...` lines with your own code. Before you run a cell that answers a "how many" question, write down your guess first.

## About the data

The `penguins` dataset comes with Seaborn. It has measurements for 344 penguins from three species, collected on three islands in the Palmer Archipelago, Antarctica.

| Column | Description |
|:--|:--|
| `species` | Adelie, Chinstrap, or Gentoo |
| `island` | Biscoe, Dream, or Torgersen |
| `bill_length_mm` | Bill length (mm) |
| `bill_depth_mm` | Bill depth (mm) |
| `flipper_length_mm` | Flipper length (mm) |
| `body_mass_g` | Body mass (g) |
| `sex` | Male or Female |

## 0. Setup

In [ ]:
# Import the libraries we use in this lab
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

## 1. Load and inspect the data

*Review from before Lecture 07: the code is filled in. Run the cells and skim the output.*

Load the dataset into a DataFrame called `df` and take a first look.

In [ ]:
# Load the penguins dataset with sns.load_dataset() and save it as df
df = sns.load_dataset('penguins')

# Show the first 5 rows
df.head()

In [ ]:
# Print the shape of df
print(df.shape)

# Use .info() to see column types and non-null counts
df.info()

In [ ]:
# Count the penguins per species with .value_counts()
df['species'].value_counts()

> **Discuss:** Which columns are numeric and which are categorical? Is the dataset balanced across species?

**Answer:** The four measurements (`bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g`) are numeric (`float64`). `species`, `island` and `sex` are categorical (stored as text). It is not balanced: 152 Adelie, 124 Gentoo and only 68 Chinstrap.

## 2. Clean the data

Real data is rarely complete. Let's find the gaps and decide what to do with them. The missing-value cells are review and already filled in; the "numbers stored as text" exercise after them is new.

In [ ]:
# Count the missing values in each column
# Hint: .isnull() followed by .sum()
df.isnull().sum()

There are only a few missing rows, so we will simply drop them. Save the result as `penguins` and check how many rows are left.

In [ ]:
# Drop the rows with any missing values and save the result as penguins
penguins = df.dropna()

# Print the new shape and confirm there are no missing values left
print(penguins.shape)
print(penguins.isnull().sum().sum())

> **Discuss:** Dropping rows is easy here. When would dropping rows be a bad idea? What else could we do?

**Answer:** We lost 11 of 344 rows (about 3%), which is fine. Dropping is a bad idea when many rows are missing, or when the missing rows are different from the rest (for example, if mostly one species had no `sex` recorded), because the data that is left would no longer represent the population. Alternatives: fill the gap (`.fillna()` with the mean, median or a label such as 'Unknown'), or drop only the column that has most of the gaps.

**Numbers stored as text.** A field team typed some weights into a spreadsheet and left one cell as a blank space. Run the cell below to create their table.

In [ ]:
# Run this cell to create the field notes
field_notes = pd.DataFrame({
    'penguin_id': [1, 2, 3, 4, 5],
    'weight_g': ['3750', '3800', ' ', '3250', '4100']
})
field_notes.dtypes

In [ ]:
# 1. Try converting weight_g with .astype('float64'). Read the error message, then comment the line out.
# field_notes['weight_g'].astype('float64')   # ValueError: could not convert string to float: ' '

# 2. Convert it with pd.to_numeric(..., errors='coerce') instead and save it back to the column
field_notes['weight_g'] = pd.to_numeric(field_notes['weight_g'], errors='coerce')

# 3. Print the mean weight. How many weights went into that mean?
print(field_notes['weight_g'].mean())
print(field_notes['weight_g'].count())

> **Discuss:** Why did `.astype('float64')` fail, and what did `errors='coerce'` do with the blank cell?

## 3. Summarize with `groupby` and `pivot_table`

What does an "average" penguin of each species look like?

In [ ]:
# Group by species and compute the mean of the four numeric columns
# Hint: select the numeric columns first, or use numeric_only=True
penguins.groupby('species').mean(numeric_only=True)

A pivot table gives the same kind of summary laid out as a grid: `index` sets the rows, `columns` sets the columns, and `aggfunc` decides what goes in each cell.

In [ ]:
# Pivot table of the mean body_mass_g, with species as rows and sex as columns
# ...

In [ ]:
# Now count the penguins on each island: species as rows, island as columns, aggfunc='count'
# ...

> **Discuss:** Which species is the heaviest, and is the male/female gap similar across species? In the island table, why are some cells `NaN` instead of 0?

**Shares.** What share of all penguins are Gentoo? Watch the denominator: it should be *all* penguins, not just the non-Gentoo ones.

In [ ]:
# 1. Compute the share of Gentoo penguins as (number of Gentoo) / (number of penguins)
# ...

# 2. Check your answer with value_counts(normalize=True)
# ...

## 4. Visualize

Every plot needs a title and readable axis labels.

**Distribution.** Plot a histogram of `body_mass_g`, with one color per species.

In [ ]:
# Use sns.histplot() with x='body_mass_g' and hue='species'
# ...

# Add a title and axis labels
# ...

**Comparing groups.** Make a box plot of `flipper_length_mm` for each species, split by sex.

In [ ]:
# Use sns.boxplot() with species on the x-axis, flipper_length_mm on the y-axis and hue='sex'
# ...

# Add a title and axis labels
# ...

> **Discuss:** How many boxes are there? What do the line inside a box, the box itself, the whiskers and any separate dots show? Does a taller box mean more penguins?

**Relationships.** Make a scatter plot of `flipper_length_mm` (x) against `body_mass_g` (y), colored by species.

In [ ]:
# Use sns.scatterplot() with hue='species'
# ...

# Add a title and axis labels
# ...

**Several plots in one figure.** `fig, axes = plt.subplots(nrows, ncols)` creates a grid of Axes. With more than one row, pick a plot with `axes[row, col]`, counting from 0.

In [ ]:
# 1. Create a 2x2 grid with plt.subplots(2, 2, figsize=(10, 8))
# ...

# 2. Draw a histogram of each numeric column in its own Axes, using the ax= argument of sns.histplot()
#    Top-left: bill_length_mm, top-right: bill_depth_mm, bottom-left: flipper_length_mm, bottom-right: body_mass_g
# ...

# 3. Give each Axes a title with .set_title()
# ...

# 4. Limit the x-axis of the bottom-right plot to 2,500-6,500 g with .set_xlim()
# ...

# plt.tight_layout() keeps the titles from overlapping
# ...

> **Discuss:** How many Axes does `plt.subplots(2, 2)` create, and what would `axes[0, 2]` do here? Which chart would you use to show what share of the penguins each species makes up?

## 5. Combine tables with `pd.merge`

Data often lives in more than one table. Below is a lookup table from a different source. Look at it closely before you merge: it uses a different column name for the key, it has no Chinstrap row, and it has a species we never measured.

In [ ]:
# Run this cell to create the lookup table
species_info = pd.DataFrame({
    'Species': ['Adelie', 'Gentoo', 'Emperor'],
    'scientific_name': ['Pygoscelis adeliae', 'Pygoscelis papua', 'Aptenodytes forsteri']
})
species_info

In [ ]:
# 1. Try pd.merge(penguins, species_info, on='species', how='left'). Read the error, then comment the line out.
# ...

# 2. Fix it with left_on= and right_on=, and save the result as penguins_full
# ...

# 3. Print the shape and count the missing scientific names
# ...

Before you run the next cell, write down how many rows you expect for an **inner** and for an **outer** join. Use the species counts from Section 1 (after cleaning: 146 Adelie, 68 Chinstrap, 119 Gentoo).

In [ ]:
# Print the number of rows for how='inner' and for how='outer'
# ...

> **Discuss:** Explain the row counts for left, inner and outer.

**Anti-join.** An anti-join keeps the rows of one table that have *no* match in the other. In pandas we use `.isin()` with `~` ("not").

In [ ]:
# Keep the penguins whose species is NOT in species_info['Species']
# Hint: penguins[~penguins['species'].isin(...)]
# ...

**Duplicate keys.** Suppose someone entered the Gentoo row twice in the lookup table.

In [ ]:
# 1. Create species_info_dup by stacking species_info with an extra copy of the Gentoo row
#    Hint: pd.concat([species_info, species_info[species_info['Species'] == 'Gentoo']])
# ...

# 2. Left-merge penguins with species_info_dup and print the number of rows. Guess first!
# ...

> **Discuss:** Why did the number of rows change?

## 6. Time series

A webcam at a colony counts the penguins it sees each day. The cell below creates one value per day for 2024 (a leap year), with more penguins in the southern summer.

In [ ]:
# Run this cell to create the daily counts
rng = np.random.default_rng(0)
dates = pd.date_range('2024-01-01', '2024-12-31', freq='D')
day = np.arange(len(dates))
counts = 200 + 120 * np.cos(2 * np.pi * day / 366) + rng.normal(0, 30, len(dates))
colony = pd.Series(counts.round(), index=dates, name='penguins_seen')
colony.head()

In [ ]:
# 1. How many daily values are there?
# ...

# 2. Down-sample to monthly averages with .resample('ME').mean(). How many rows does it have?
# ...

# 3. Down-sample to weekly totals with .resample('W').sum(). How many rows?
# ...

In [ ]:
# 4. Compute a 30-day rolling mean with .rolling(30).mean(). How many rows? How many NaN values at the start?
# ...

# 5. Plot the daily series and the rolling mean on the same Axes, with a legend
# ...

In [ ]:
# 6. Use .shift(1) to put yesterday's count next to today's, and compute the day-over-day change
#    Show the first 3 rows of a DataFrame with columns today, yesterday and change
# ...

> **Discuss:** Down-sampling, a rolling window and shifting all change the series. Which ones change the number of rows?

## 7. Machine learning: one classification model

**Which kind of task?** For each question, decide whether it is regression, classification or clustering:
1. Predict a penguin's body mass in grams from its flipper length.
2. Predict whether a penguin is male or female from its measurements.
3. Without using `species`, find natural groups of penguins with similar measurements.

> **Discuss:** Write down your three answers before moving on.

Now we build a classifier for one yes/no question: **is this penguin a Chinstrap?** Only about 1 in 5 penguins is a Chinstrap, so watch what happens to accuracy.

In [ ]:
# Run this cell to import what we need and create the 0/1 target
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix

penguins = penguins.copy()
penguins['is_chinstrap'] = (penguins['species'] == 'Chinstrap').astype(int)
penguins['is_chinstrap'].value_counts()

In [ ]:
# 1. Create X with body_mass_g only (double brackets) and y with is_chinstrap
# ...

# 2. Split into train and test sets with test_size=0.3 and random_state=42
# ...

# 3. How many rows are in the training set and in the test set?
# ...

In [ ]:
# 4. Fit a GaussianNB model on the training data and predict on the test set
# ...

# 5. Print the accuracy and the confusion matrix
# ...

In `confusion_matrix(y_test, y_pred)` the rows are the true classes and the columns are the predicted classes, both in the order 0, 1:

|  | Predicted 0 | Predicted 1 |
|:--|:--|:--|
| **True 0** | TN (true negative) | FP (false positive) |
| **True 1** | FN (false negative) | TP (true positive) |

- **Accuracy** = (TP + TN) / all
- **Sensitivity** (recall) = TP / (TP + FN): the share of real Chinstraps the model catches
- **Specificity** = TN / (TN + FP): the share of non-Chinstraps the model correctly clears

In [ ]:
# 6. Get tn, fp, fn, tp with confusion_matrix(y_test, y_pred).ravel(), then compute sensitivity and specificity
# ...

# 7. Baseline: what accuracy would we get by predicting 0 ("not a Chinstrap") for every penguin in the test set?
# ...

> **Discuss:** Is this a good model? Compare its accuracy with the baseline and look at its sensitivity.

In [ ]:
# 8. Repeat steps 1-6 with two features: bill_length_mm and body_mass_g. What changes?
# ...

> **Discuss:** Why did adding bill length help so much?

## Wrap-up

In one or two sentences each:

1. Which step told you the most about this data before any modeling?
2. When does a merge change the number of rows, and how can you check for it?
3. Why is accuracy alone not enough to judge a classifier?